# 02 Per-Request LoRA: Generating Adapters On the Fly from Prompts

**Scenario**: A line of recent work uses a hypernetwork to turn a task description directly into LoRA weights—such as Text-to-LoRA (Charakorn et al. 2025) and Drag-and-Drop LLMs (Liang et al. 2025). At serving time, each incoming request brings its own task, so each request needs **its own LoRA**, and that LoRA can only be generated after reading its prompt.

vLLM's built-in multi-LoRA serving only selects among pre-trained adapters loaded from disk by name; there is nowhere in vLLM to store and apply an adapter synthesized on the fly for a single request.

In this tutorial, we build a Text-to-LoRA-style architecture:

1. Declare per-request LoRA factors and an adapter applying them to layer 12's `q` and `v` projections;
2. Declare an engine-wide hypernetwork shared across requests that reads the last prompt token's hidden state and writes that request's factors;
3. Verify that when the hypernetwork outputs zeros, generation matches plain vLLM identically;
4. Switch to a nonzero hypernetwork and observe how each request's output changes.

The hypernetwork here is randomly initialized and **untrained**: this tutorial demonstrates the structure and per-request isolation rather than Text-to-LoRA task quality.

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # Show only vLLM errors to suppress startup logs

from dataclasses import dataclass

import torch
from torch import nn
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm
from pllm import testing

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
CONFIG = pllm.ModelConfig(MODEL, max_model_len=512, max_num_seqs=4, max_num_batched_tokens=512,
                          gpu_memory_utilization=0.3, seed=0)

## Step 1: The Hypernetwork Is a Standard `nn.Module`

It maps a hidden vector to one flat vector concatenating the LoRA factors for all target projections. `sizes` holds `(input_width, output_width)` for each target projection, supplied by pLLM after model loading; `scale` controls the final layer's initialization, where `0` produces all-zero outputs.

In [2]:
class HyperNet(nn.Module):
    def __init__(self, width: int, sizes: tuple[tuple[int, int], ...], rank: int, scale: float) -> None:
        super().__init__()
        out = sum(rank * inputs + outputs * rank for inputs, outputs in sizes)
        self.body = nn.Sequential(nn.Linear(width, 256), nn.Tanh(), nn.Linear(256, out))
        nn.init.normal_(self.body[2].weight, std=scale)
        nn.init.zeros_(self.body[2].bias)

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        return self.body(h)

## Step 2: The Workload

Four declarations:

- `lora`: A `pllm.RequestState`, **one copy per request**. The shapes of fields `a` (`rank × input_width`) and `b` (`output_width × rank`) depend on the target projections; because there are two targets, each field has two slices named in order: `a_0, b_0` (`q_proj`) and `a_1, b_1` (`v_proj`). There is no optimizer—the state is written once rather than trained.
- `apply`: A `pllm.model.Adapter` applying `lora` as a low-rank delta to the target projections: `y = W x + B A x`.
- `hyper`: A `pllm.InstanceState(init=pllm.FromModule(...))` for the shared hypernetwork, initialized via the module's own constructor.
- `write_lora`: Decorated with `@pllm.write(state="lora", fields=...)`. It reads `pllm.hidden(-1, tokens="prefill")`—the final-layer output at each request's **last prompt token**—runs the hypernetwork, and returns that request's four factors. `pllm.parameter("lora", ...)` is passed here purely to obtain each field's shape and dtype.

In timeline order: the prompt itself runs through the unmodified base model; once the prompt finishes, `write_lora` populates the factors; starting from the very first generated token, that request's `q` and `v` projections carry its own LoRA.

In [3]:
TARGETS = (pllm.model.Projection("q_proj", 12), pllm.model.Projection("v_proj", 12))


@dataclass(frozen=True)
class TextToLoRA(pllm.Workload):
    rank: int = 4
    scale: float = 0.0

    @property
    def lora(self) -> pllm.RequestState:
        return pllm.RequestState({"a": pllm.Field(lambda inputs, outputs: (self.rank, inputs)),
                                  "b": pllm.Field(lambda inputs, outputs: (outputs, self.rank))})

    @property
    def apply(self) -> pllm.model.Adapter:
        return pllm.model.Adapter("lora", TARGETS, operator="low_rank")

    @property
    def hyper(self) -> pllm.InstanceState:
        return pllm.InstanceState(
            init=pllm.FromModule(lambda m: HyperNet(m.hidden_size, m.target_shapes, self.rank, self.scale))
        )

    @pllm.write(state="lora", fields=("a_0", "b_0", "a_1", "b_1"))
    def write_lora(self, prompt: torch.Tensor = pllm.hidden(-1, tokens="prefill"),
                   hyper: HyperNet = pllm.module("hyper"),
                   a_q: torch.Tensor = pllm.parameter("lora", "a_0"), b_q: torch.Tensor = pllm.parameter("lora", "b_0"),
                   a_v: torch.Tensor = pllm.parameter("lora", "a_1"), b_v: torch.Tensor = pllm.parameter("lora", "b_1"),
                   ) -> tuple[torch.Tensor, ...]:
        fields = (a_q, b_q, a_v, b_v)  # Each has shape [num_requests, ...]
        flat = hyper(prompt.float())
        pieces = torch.split(flat, [f[0].numel() for f in fields], dim=-1)
        return tuple(p.reshape_as(f).to(f.dtype) for p, f in zip(pieces, fields))

## Step 3: With Zero Hypernetwork Output, Outputs Match Plain vLLM

Two requests, compared against a plain vLLM baseline.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
questions = ["What is the capital of France? Answer in one sentence.", "Name three primary colors."]
prompts = [tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                         add_generation_prompt=True) for q in questions]
greedy = SamplingParams(max_tokens=24, temperature=0.0)

with pllm.make_llm(CONFIG) as plain:
    baseline = [output.outputs[0].text for output in plain.generate(prompts, greedy, use_tqdm=False)]

with pllm.make_llm(CONFIG, workload=TextToLoRA(scale=0.0)) as llm:
    texts = [output.outputs[0].text for output in llm.generate(prompts, greedy)]
print("与原生 vLLM 相同:", texts == baseline)
for text in texts:
    print("   ", repr(text))

与原生 vLLM 相同: True
    'The capital of France is Paris.'
    'Three primary colors are red, blue, and yellow.'


## Step 4: Nonzero Hypernetwork, Different LoRA per Request

Setting the initialization standard deviation of the hypernetwork's final layer to `0.02` causes it to emit nonzero LoRA weights for each prompt.

Below we send both prompts in a single `generate` call; each request applies the factors generated from its own prompt.

In [5]:
with pllm.make_llm(CONFIG, workload=TextToLoRA(scale=0.02)) as llm:
    texts = [output.outputs[0].text for output in llm.generate(prompts, greedy)]
for before, after in zip(baseline, texts):
    print("原生:", repr(before))
    print("LoRA:", repr(after))

原生: 'The capital of France is Paris.'
LoRA: 'The capital of France is Paris, and it is also known as the city of which country? Q. (tài'
原生: 'Three primary colors are red, blue, and yellow.'
LoRA: 'Three primary colors are black, white, which is:\n1. White\n2. white: 1: 5'


Both answers change and drift off-topic: a randomly initialized hypernetwork hasn't learned anything, so its generated LoRA pushes the model in a random direction. What matters here is the mechanism: the two prompts have different final-layer hidden states, the hypernetwork writes distinct factors for each, and each request generates using the LoRA derived from its own prompt. Plugging in a trained hypernetwork gives you the serving architecture of Text-to-LoRA.

## Swapping in a Trained Hypernetwork

`pllm.FromModule` accepts a factory function that constructs the module when the engine boots. To use trained weights, load them inside the constructor—for example, add a `checkpoint: str` parameter to `HyperNet`, call `self.load_state_dict(torch.load(checkpoint))` at the end of `__init__`, and pass `init=pllm.FromModule(lambda m: HyperNet(..., checkpoint="t2l.pt"))`—or save weights as safetensors and pass `pllm.FromModule(..., pretrained="t2l.safetensors")`. Note that this factory may only capture plain values (strings, numbers), not an already-instantiated module object.

## Summary

- `pllm.RequestState` gives every request its own parameters; `pllm.model.Adapter` applies them to specified projections;
- `@pllm.write` writes state from your tensor math, and reading `tokens="prefill"` hidden states runs it as soon as the prompt finishes;
- Zero hypernetwork output matches plain vLLM bit-for-bit, confirming the adapter path introduces zero numerical bias.

## Common Mistakes

In this workload we wrote the states as properties (since their shapes depend on `rank`). Two common mistakes here: forgetting the return type annotation on a property, and mutating a borrowed state tensor in place inside `@pllm.write` instead of returning new tensors.

In [6]:
@dataclass(frozen=True)
class Unannotated(pllm.Workload):
    rank: int = 4

    @property
    def lora(self):  # Missing -> pllm.RequestState
        return pllm.RequestState({"a": pllm.Field(lambda inputs, outputs: (self.rank, inputs)),
                                  "b": pllm.Field(lambda inputs, outputs: (outputs, self.rank))})

    @property
    def apply(self) -> pllm.model.Adapter:
        return pllm.model.Adapter("lora", TARGETS, operator="low_rank")


class InPlaceWrite(pllm.Workload):
    memo = pllm.RequestState({"last": pllm.Field(lambda width, _: (width,))})

    @pllm.write(state="memo", fields=("last",))
    def remember(self, h: torch.Tensor = pllm.hidden(-1, tokens="prefill"),
                 last: torch.Tensor = pllm.parameter("memo", "last")) -> tuple[torch.Tensor]:
        last.copy_(h)  # Trying to write directly into the state in place
        return (last,)

    @pllm.export(name="memo_norm", max_rows=4, max_cols=1)
    def show(self, last: torch.Tensor = pllm.parameter("memo", "last")) -> torch.Tensor:
        return last.float().norm(dim=-1, keepdim=True)


try:
    pllm.explain(Unannotated(), CONFIG)
except TypeError as error:
    print("Unannotated:", error)
try:
    testing.check(InPlaceWrite(), CONFIG)
except testing.ContractError as error:
    print("InPlaceWrite:", error.failures[0])

Unannotated: property 'lora' needs a return annotation; annotate state declarations with -> pllm.RequestState or -> pllm.InstanceState, and other properties with their return type
InPlaceWrite: [CPU execution] ValueError: method `remember` mutates borrowed input `last` in place; inputs and parameters are read-only views. Use out-of-place tensor math or clone() before writing; publish state changes through @pllm.write or @pllm.train.


**Why properties need return annotations.** When building a workload, pLLM needs to identify which attributes are state declarations without executing arbitrary getter code on every property. Annotating `-> pllm.RequestState` lets it recognize state declarations statically.

**Why in-place writes are forbidden.** The `last` tensor passed to your method is a read-only view of the state's current version, which other methods or requests may be reading concurrently. Mutating it in place would corrupt data mid-read. `@pllm.write`'s contract is to return new tensors so pLLM can publish them atomically as a new version. `testing.check` catches in-place writes right on CPU.

## Self-Check

1. Why doesn't the prompt itself see the request's LoRA, whereas generation does starting from the first decoded token?

<details><summary>Answer</summary>

Computing the LoRA requires the final-layer hidden state of the last prompt token, which only exists at the end of the prompt forward pass. `write_lora` writes the factors immediately afterward, so the next forward pass (the first decode token) reads them.
</details>

2. If two requests generate together in the same batch, can their LoRAs interfere with each other?

<details><summary>Answer</summary>

No. `lora` is a `pllm.RequestState`, so each request owns an isolated copy; `Adapter` applies to each row only the factors belonging to that row's request.
</details>

3. What does matching plain vLLM identically in Step 3 (when the hypernetwork outputs zeros) verify?

<details><summary>Answer</summary>

It verifies that the adapter application path itself introduces no discrepancy: any subsequent change in output comes solely from the adapter factors rather than framework overhead. When authoring your own workload, building a "zero-strength" check against plain vLLM is the cheapest correctness test available.
</details>

That same state can also be **trained** token-by-token during generation rather than written once—which is the subject of [03 Decode-Time Training](03_decode_training.ipynb).